# Crop-Type Classification — Exploration & Training

Multi-temporal **Sentinel-2** crop classification on a subset of the public
**PASTIS** benchmark (102 patches, tile `t31tfm`, southern France).

This notebook is a narrative walkthrough of the pipeline implemented in `src/`:
data loading → EDA → splits → feature engineering → Random-Forest training →
evaluation → output visualisation. The full analysis is in
[`report.md`](../report.md).

> Run from the project root, or the path setup cell below will add it to
> `sys.path`. Heavy steps reuse artefacts already saved under `outputs/`.

In [1]:
import os, sys, json
# Ensure the project root (containing `src/`) is importable.
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image  # only to display pre-rendered figures inline

from src.data_loading import (load_config, DataPaths, list_patch_ids,
                              load_s2, load_target, load_metadata, describe_patch)
from src import preprocessing as pp
from src import visualization as viz

cfg = load_config("configs/config.yaml")
paths = DataPaths.from_config(cfg)
label_names = {int(k): v for k, v in cfg["labels"]["names"].items()}
FIG = cfg["paths"]["figures"]

def show(path, w=14):
    if not os.path.exists(path):
        print("missing:", path); return
    img = np.asarray(Image.open(path))
    plt.figure(figsize=(w, w * img.shape[0] / img.shape[1]))
    plt.imshow(img); plt.axis("off"); plt.show()

print("Project root:", ROOT)

Project root: D:\task\Agri-Assignment_01\Agri-Assignment_01


## 1. Data loading & inspection

Each Sentinel-2 patch is `(T, C, H, W)` = `(46, 10, 128, 128)` int16; each target
is `(1, 128, 128)` uint8 with class IDs `0` (background), `1–18` (crops), `19`
(void/border, ignored).

In [2]:
ids = list_patch_ids(paths)
print(f"{len(ids)} patches available. First five: {ids[:5]}")
info = describe_patch(paths, ids[0])
print(json.dumps(info, indent=2)[:500])

102 patches available. First five: [30003, 30013, 30014, 30023, 30026]
{
  "patch_id": 30003,
  "s2_shape": [
    46,
    10,
    128,
    128
  ],
  "s2_dtype": "int16",
  "target_shape": [
    128,
    128
  ],
  "classes": [
    0,
    1,
    2,
    3,
    4,
    5,
    12,
    19
  ],
  "class_pixel_counts": {
    "0": 5458,
    "1": 4045,
    "2": 2631,
    "3": 1484,
    "4": 1441,
    "5": 425,
    "12": 88,
    "19": 812
  }
}


## 2. Exploratory data analysis

We reuse `src.run_eda` (already executed) to render figures. If they are not on
disk yet, uncomment the run cell.

In [3]:
# from src.run_eda import main as run_eda; run_eda()   # (re)generate EDA figures
show(os.path.join(FIG, "class_distribution.png"))

C:\Users\hakimali.datardi\AppData\Local\Temp\ipykernel_52984\2372005705.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(img); plt.axis("off"); plt.show()


**Severe class imbalance** — background, meadow, wheat, corn and soybeans
dominate; grapevine, orchard, potatoes, durum wheat are each < 1 % of pixels.

In [4]:
show(os.path.join(FIG, "ndvi_profiles.png"))

C:\Users\hakimali.datardi\AppData\Local\Temp\ipykernel_52984\2372005705.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(img); plt.axis("off"); plt.show()


**Phenology is the signal.** Classes trace distinct seasonal NDVI curves.
The synchronous dips across *all* classes (indices ~3, 6, 26, 36) are
**cloud-contaminated dates** — the motivation for using temporal
median/percentile features that are robust to them.

In [5]:
# Example patch: Sentinel-2 RGB next to ground truth (least-cloudy frame)
pid = ids[0]
s2 = load_s2(paths, pid)
t = pp.least_cloudy_timestep(s2, cfg["preprocess"]["reflectance_scale"])
viz.plot_patch_overview(s2, load_target(paths, pid), t, label_names,
                        os.path.join(FIG, f"nb_overview_{pid}.png"), title=f"Patch {pid}")
show(os.path.join(FIG, f"nb_overview_{pid}.png"))

C:\Users\hakimali.datardi\AppData\Local\Temp\ipykernel_52984\2372005705.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(img); plt.axis("off"); plt.show()


In [6]:
show(os.path.join(FIG, "aoi_footprints.png"), w=8)

C:\Users\hakimali.datardi\AppData\Local\Temp\ipykernel_52984\2372005705.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(img); plt.axis("off"); plt.show()


## 3. Train / validation / test split

We reuse the **official PASTIS 5-fold** assignment from `metadata.geojson`,
splitting at the **patch** level (folds 1–3 train, 4 val, 5 test) so no pixels
leak between train and evaluation. IDs are saved under `splits/`.

In [7]:
from src.splits import make_splits, write_splits, read_split
splits = make_splits(cfg); write_splits(splits, cfg["paths"]["splits"])
for k, v in splits.items():
    print(f"{k:5s}: {len(v)} patches")

train: 66 patches
val  : 18 patches
test : 18 patches


## 4. Feature engineering

Per pixel we build **112 features**: 8 temporal statistics
(`mean/std/min/max/p25/p50/p75/amplitude`) of each of the 10 bands **and** of
NDVI/NDWI/NDRE/EVI. `(T,C,H,W)` → `(H*W, 112)`.

In [8]:
stack, names = pp.build_feature_stack(load_s2(paths, ids[0]), cfg)
X = pp.stack_to_pixels(stack)
print("feature stack:", stack.shape, "-> pixel matrix:", X.shape)
print("example features:", names[:4], "...", names[-2:])

feature stack: (112, 128, 128) -> pixel matrix: (16384, 112)
example features: ['B2_mean', 'B2_std', 'B2_min', 'B2_max'] ... ['evi_p75', 'evi_amplitude']


## 5. Model training

Random Forest (300 trees, `class_weight=balanced_subsample`) on class-capped
sampled pixels. Training takes ~80 s on CPU. If a model is already saved we skip
refitting.

In [9]:
import joblib
model_path = os.path.join(cfg["paths"]["models"], "rf_model.joblib")
if os.path.exists(model_path):
    print("Loading existing model:", model_path)
    model = joblib.load(model_path)
else:
    from src.train import main as train_main
    train_main("configs/config.yaml")
    model = joblib.load(model_path)
print("n_features:", model.n_features_in_, "| n_classes:", len(model.classes_))

Loading existing model: outputs/models\rf_model.joblib


n_features: 112 | n_classes: 18


## 6. Evaluation

Dense evaluation (all valid pixels) on validation and test splits.

In [10]:
from src.evaluate import evaluate_split, print_report, _labels_for_eval
labels = _labels_for_eval(cfg)
for split in ["val", "test"]:
    m = evaluate_split(cfg, split)
    print(f"[{split}] acc={m['overall_accuracy']:.4f} "
          f"macroF1={m['macro_f1']:.4f} mIoU={m['mean_iou']:.4f}")

[val] acc=0.7337 macroF1=0.3834 mIoU=0.2636


[test] acc=0.7448 macroF1=0.3686 mIoU=0.2729


In [11]:
# Detailed per-class report on the test split
from src.data_loading import DataPaths
from src.dataset import build_dataset
rng = np.random.default_rng(cfg["seed"])
X_te, y_te, _ = build_dataset(paths, read_split(cfg["paths"]["splits"], "test"),
                              cfg, sample=False, rng=rng)
print_report(y_te, model.predict(X_te), labels, label_names)

                           precision    recall  f1-score   support

               Background       0.86      0.75      0.80     96924
                   Meadow       0.64      0.83      0.72     54359
        Soft winter wheat       0.80      0.76      0.78     35388
                     Corn       0.72      0.77      0.74     31525
            Winter barley       0.60      0.51      0.55     11780
          Winter rapeseed       0.90      0.90      0.90     17328
            Spring barley       0.00      0.00      0.00       153
                Sunflower       0.97      0.14      0.24      2851
                Grapevine       0.00      0.00      0.00         0
                     Beet       0.00      0.00      0.00         0
         Winter triticale       0.00      0.00      0.00      1612
       Winter durum wheat       0.00      0.00      0.00       306
Fruits/vegetables/flowers       0.00      0.00      0.00      2643
                 Potatoes       0.00      0.00      0.00     

In [12]:
show(os.path.join(FIG, "confusion_matrix_test.png"), w=9)

C:\Users\hakimali.datardi\AppData\Local\Temp\ipykernel_52984\2372005705.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(img); plt.axis("off"); plt.show()


Confusions are **agronomically sensible**: winter durum wheat and triticale
are absorbed into soft winter wheat (near-identical winter-cereal phenology);
rare classes with tiny support collapse to zero recall.

In [13]:
show(os.path.join(FIG, "feature_importances.png"), w=8)

C:\Users\hakimali.datardi\AppData\Local\Temp\ipykernel_52984\2372005705.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(img); plt.axis("off"); plt.show()


SWIR (B11/B12) and red-edge (B5/B6) seasonality plus NDVI/NDRE percentiles
carry the most information — consistent with moisture and canopy-chlorophyll
dynamics distinguishing crops.

## 7. Output visualisation

Sentinel-2 RGB | ground truth | prediction, for test patches.

In [14]:
for pid in read_split(cfg["paths"]["splits"], "test")[:3]:
    show(os.path.join(FIG, f"prediction_{pid}.png"))

C:\Users\hakimali.datardi\AppData\Local\Temp\ipykernel_52984\2372005705.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(img); plt.axis("off"); plt.show()


The model captures large parcels well but is noisy inside fields and at
boundaries — the expected limitation of a **per-pixel** model with no spatial
context.

## 8. Spatial post-processing (accuracy boost)

Crop parcels are spatially coherent, so we apply an **unsupervised spatial
majority filter** (window selected on validation) to each prediction map. It
uses only the model's own predictions — no ground-truth — so there is no label
leakage.

In [15]:
from src.evaluate import evaluate_split_spatial
for split in ["val", "test"]:
    raw = evaluate_split(cfg, split)
    sm = evaluate_split_spatial(cfg, split)
    print(f"[{split}] per-pixel acc={raw['overall_accuracy']:.4f} mIoU={raw['mean_iou']:.4f}"
          f"  ->  +smoothing acc={sm['overall_accuracy']:.4f} mIoU={sm['mean_iou']:.4f}")

[val] per-pixel acc=0.7337 mIoU=0.2636  ->  +smoothing acc=0.7444 mIoU=0.2870


[test] per-pixel acc=0.7448 mIoU=0.2729  ->  +smoothing acc=0.7596 mIoU=0.3031


In [16]:
# RGB | ground truth | per-pixel | smoothed
for pid in read_split(cfg["paths"]["splits"], "test")[:2]:
    show(os.path.join(FIG, f"comparison_{pid}.png"))

C:\Users\hakimali.datardi\AppData\Local\Temp\ipykernel_52984\2372005705.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(img); plt.axis("off"); plt.show()


The smoothed maps are visibly cleaner and closer to the parcel structure of
the ground truth — direct evidence that the biggest remaining gain is **spatial
context**, motivating a spatio-temporal model (U-TAE / U-Net) as the next step.

## 9. Conclusion

A well-reasoned Random-Forest baseline reaches **~74.5 % test accuracy**
per-pixel and **~76.0 % with spatial smoothing**, with fully interpretable
behaviour. See [`report.md`](../report.md) for the complete analysis and
prioritised next steps (U-TAE / U-Net for spatial context, parcel-level majority
voting, explicit cloud masking, richer temporal features).